In [1]:
# ============================================================
# PART 1 (UPDATED): Setup, Config, Multi-Dataset Support
# ============================================================

!pip install ftfy regex tqdm open_clip_torch -q

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import (
    Dataset, DataLoader,
    Subset, ConcatDataset
)
from torchvision.datasets import (
    OxfordIIITPet,
    EuroSAT,
    DTD
)
import open_clip
import matplotlib.pyplot as plt
from collections import defaultdict
import copy
import warnings
import gc
import psutil
warnings.filterwarnings('ignore')

# ── Mount Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CACHE_DIR = '/content/drive/MyDrive/blend_oxfordpets_cache'
os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
print(f"Cache directory: {DRIVE_CACHE_DIR}")

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(SEED)

# ── Device ───────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available()
                      else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: "
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ── Config ───────────────────────────────────────────────────
CONFIG = {
    'num_clients'     : 10,
    'alpha_dir'       : 0.1,
    'num_shots'       : 16,
    'adapter_rank_vis': 4,
    'adapter_rank_txt': 4,
    'proj_rank'       : 16,
    'adapter_layers'  : (1, 12),
    'num_rounds'      : 25,
    'local_epochs'    : 1,
    'local_lr'        : 0.0003,
    'batch_size'      : 8,
    'temperature'     : 0.02,
    'lambda_loss'     : 0.2,
    'gamma_static'    : 0.09,
    'gamma_min'       : 0.05,
    'gamma_max'       : 0.50,
    'tau_entropy'     : 3.0,
    'tau_divergence'  : 2.0,
    'gamma_bias'      : 1.5,
    'learn_tau'       : True,
    'eval_every'      : 2,
    'weight_decay'    : 0.01,
    'dropout_rate'    : 0.1,
    'label_smoothing' : 0.1,
}

print("\n--- CONFIG ---")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


# ============================================================
# CLIP preprocessor (needed for all datasets)
# ============================================================

_, _, clip_preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='openai'
)
print("\nCLIP preprocessor loaded.")


# ============================================================
# Dataset loader factory
# ============================================================

def load_raw_dataset(name, root='/content/data_raw'):
    """
    Load raw dataset (no preprocessing).
    Returns (train_dataset, test_dataset,
             class_names, num_classes,
             train_labels, test_labels)
    """
    ds_root = os.path.join(root, name)
    os.makedirs(ds_root, exist_ok=True)

    print(f"\nLoading {name}...")

    if name == 'OxfordPets':
        train = OxfordIIITPet(
            root=ds_root, split='trainval',
            transform=None, download=True
        )
        test = OxfordIIITPet(
            root=ds_root, split='test',
            transform=None, download=True
        )
        class_names = train.classes
        num_classes  = 37
        train_labels = list(train._labels)
        test_labels  = list(test._labels)

    elif name == 'EuroSAT':
        full = EuroSAT(
            root=ds_root,
            transform=None,
            download=True
        )
        class_names = full.classes
        num_classes  = 10

        # Reproducible 80/20 split
        n_total = len(full)
        n_train = int(0.8 * n_total)
        n_test  = n_total - n_train
        train, test = torch.utils.data.random_split(
            full, [n_train, n_test],
            generator=torch.Generator().manual_seed(SEED)
        )
        # Extract labels
        all_targets  = list(full.targets)
        train_labels = [all_targets[i]
                        for i in train.indices]
        test_labels  = [all_targets[i]
                        for i in test.indices]

        # Patch for compatibility with pipeline
        train._labels    = train_labels
        test._labels     = test_labels
        train._images    = [full.samples[i][0]
                            for i in train.indices]
        test._images     = [full.samples[i][0]
                            for i in test.indices]
        train.classes    = class_names
        test.classes     = class_names

    elif name == 'DTD':
        train_raw = DTD(
            root=ds_root, split='train',
            transform=None, download=True
        )
        val_raw = DTD(
            root=ds_root, split='val',
            transform=None, download=True
        )
        test = DTD(
            root=ds_root, split='test',
            transform=None, download=True
        )

        # Combine train + val as training pool
        # (standard practice for DTD)
        train_labels_raw = [s[1] for s in
                            train_raw._samples]
        val_labels_raw   = [s[1] for s in
                            val_raw._samples]
        test_labels_raw  = [s[1] for s in
                            test._samples]

        # Create combined train object
        class CombinedDTD:
            def __init__(self, t, v):
                self._images = (
                    [s[0] for s in t._samples] +
                    [s[0] for s in v._samples]
                )
                self._labels = (
                    [s[1] for s in t._samples] +
                    [s[1] for s in v._samples]
                )
                self.classes = t.classes

            def __len__(self):
                return len(self._labels)

            def __getitem__(self, idx):
                from PIL import Image
                img = Image.open(
                    self._images[idx]
                ).convert('RGB')
                return img, self._labels[idx]

        train        = CombinedDTD(train_raw, val_raw)
        class_names  = train_raw.classes
        num_classes  = 47
        train_labels = train._labels
        test_labels  = test_labels_raw
        test._labels = test_labels_raw
        test._images = [s[0] for s in test._samples]
        test.classes = class_names

    else:
        raise ValueError(f"Unknown dataset: {name}")

    print(f"  Train: {len(train_labels)} samples")
    print(f"  Test : {len(test_labels)}  samples")
    print(f"  Classes: {num_classes}")

    return (train, test, class_names,
            num_classes, train_labels, test_labels)


# ============================================================
# Core pipeline utilities
# (same as before — work for all datasets)
# ============================================================

def sample_few_shot(labels, class_ids, n_shots, seed=42):
    rng = np.random.RandomState(seed)
    class_to_indices = defaultdict(list)
    for idx, label in enumerate(labels):
        if label in class_ids:
            class_to_indices[label].append(idx)
    selected = []
    for cls in class_ids:
        cls_indices = class_to_indices[cls]
        n       = min(n_shots, len(cls_indices))
        chosen  = rng.choice(
            cls_indices, n, replace=False
        ).tolist()
        selected.extend(chosen)
    return selected


def dirichlet_split(labels, indices, class_ids,
                    n_clients, alpha, seed=42):
    rng = np.random.RandomState(seed)
    class_to_idx = defaultdict(list)
    for idx in indices:
        class_to_idx[labels[idx]].append(idx)

    client_indices      = [[] for _ in range(n_clients)]
    client_class_counts = [
        {c: 0 for c in class_ids}
        for _ in range(n_clients)
    ]

    for cls in class_ids:
        cls_idx = np.array(class_to_idx[cls])
        if len(cls_idx) == 0:
            continue
        rng.shuffle(cls_idx)
        proportions = rng.dirichlet(
            alpha * np.ones(n_clients)
        )
        counts      = (proportions * len(cls_idx)
                       ).astype(int)
        counts[-1]  = len(cls_idx) - counts[:-1].sum()
        counts      = np.maximum(counts, 0)
        start = 0
        for cid in range(n_clients):
            end = start + counts[cid]
            client_indices[cid].extend(
                cls_idx[start:end].tolist()
            )
            client_class_counts[cid][cls] = int(
                counts[cid]
            )
            start = end

    return client_indices, client_class_counts


def compute_client_entropy(class_counts):
    counts = np.array(
        [v for v in class_counts.values()],
        dtype=float
    )
    counts = counts[counts > 0]
    if len(counts) == 0:
        return 1.0
    total  = counts.sum()
    probs  = counts / total
    H      = -np.sum(probs * np.log(probs + 1e-10))
    H_max  = np.log(len(counts))
    if H_max < 1e-10:
        return 0.0
    return float(H / H_max)


def build_label_remap(client_class_counts):
    remaps            = []
    local_class_lists = []
    for counts in client_class_counts:
        present = sorted(
            [c for c, n in counts.items() if n > 0]
        )
        remap   = {c: i for i, c in enumerate(present)}
        remaps.append(remap)
        local_class_lists.append(present)
    return remaps, local_class_lists


def remap_labels(labels_tensor, remap_dict, device):
    return torch.tensor(
        [remap_dict.get(l.item(), -1)
         for l in labels_tensor],
        dtype=torch.long, device=device
    )


# ============================================================
# Drive-backed tensor dataset (unchanged)
# ============================================================

class DriveTensorDataset(Dataset):
    def __init__(self, cache_dir, label_remap=None):
        self.label_remap = label_remap
        img_path = os.path.join(cache_dir, 'images.pt')
        lbl_path = os.path.join(cache_dir, 'labels.pt')
        self.images = torch.load(img_path)
        self.labels = torch.load(lbl_path)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img   = self.images[idx].float()
        label = self.labels[idx].item()
        if self.label_remap is not None:
            label = self.label_remap.get(label, -1)
        return img, label


# ============================================================
# Preprocess and save to Drive (unchanged)
# ============================================================

def preprocess_and_save_split(
    raw_dataset, indices, labels_list,
    save_path, clip_preprocess, desc=""
):
    os.makedirs(save_path, exist_ok=True)
    img_file = os.path.join(save_path, 'images.pt')
    lbl_file = os.path.join(save_path, 'labels.pt')

    if os.path.exists(img_file) and \
            os.path.exists(lbl_file):
        print(f"  [{desc}] Cached ✓")
        return

    print(f"  [{desc}] Preprocessing "
          f"{len(indices)} images...")

    from PIL import Image
    img_paths = raw_dataset._images

    all_imgs   = []
    all_labels = []

    for i, idx in enumerate(indices):
        img_path = img_paths[idx]
        label    = labels_list[idx]
        img      = Image.open(
            img_path
        ).convert('RGB')
        tensor   = clip_preprocess(img)
        all_imgs.append(tensor.half())
        all_labels.append(label)
        img.close()
        if (i+1) % 200 == 0 or (i+1) == len(indices):
            print(f"    {i+1}/{len(indices)}", end='\r')

    print()
    imgs_t   = torch.stack(all_imgs)
    labels_t = torch.tensor(
        all_labels, dtype=torch.long
    )
    torch.save(imgs_t,   img_file)
    torch.save(labels_t, lbl_file)

    mb = imgs_t.element_size() * \
         imgs_t.nelement() / 1e6
    print(f"  [{desc}] Saved {imgs_t.shape} ({mb:.1f} MB)")

    del all_imgs, all_labels, imgs_t, labels_t
    gc.collect()


# ============================================================
# RAM check
# ============================================================

def print_ram(label=""):
    p    = psutil.Process(os.getpid())
    mb   = p.memory_info().rss / 1e6
    used = psutil.virtual_memory().used / 1e9
    tot  = psutil.virtual_memory().total / 1e9
    print(f"RAM [{label}]: "
          f"Process={mb:.0f}MB  "
          f"System={used:.1f}/{tot:.1f}GB")

print("\nPart 1 complete.")
print_ram("Part 1 end")

# ============================================================
# PART 1 END: Pre-load OxfordPets context
# so Parts 2, 3, 4 can run immediately
# without waiting for Part 5
# ============================================================

print("\nPre-loading OxfordPets context for Parts 2-4...")

# Load raw OxfordPets
(raw_train_oxford, raw_test_oxford,
 CLASS_NAMES, NUM_CLASSES,
 train_labels, test_labels) = load_raw_dataset('OxfordPets')

# Base/novel split
SPLIT_FILE = os.path.join(
    DRIVE_CACHE_DIR, 'OxfordPets_class_split.pt'
)
if os.path.exists(SPLIT_FILE):
    split         = torch.load(SPLIT_FILE)
    BASE_CLASSES  = split['base']
    NOVEL_CLASSES = split['novel']
    print(f"  Split loaded: "
          f"{len(BASE_CLASSES)} base / "
          f"{len(NOVEL_CLASSES)} novel")
else:
    rng           = random.Random(SEED)
    all_ids       = list(range(NUM_CLASSES))
    rng.shuffle(all_ids)
    nb            = NUM_CLASSES // 2
    BASE_CLASSES  = sorted(all_ids[:nb])
    NOVEL_CLASSES = sorted(all_ids[nb:])
    torch.save(
        {'base': BASE_CLASSES, 'novel': NOVEL_CLASSES},
        SPLIT_FILE
    )
    print(f"  Split created: "
          f"{len(BASE_CLASSES)} base / "
          f"{len(NOVEL_CLASSES)} novel")

# Few-shot + Dirichlet split
few_shot_idx = sample_few_shot(
    train_labels, BASE_CLASSES, CONFIG['num_shots']
)
client_indices, client_class_counts = dirichlet_split(
    train_labels, few_shot_idx,
    BASE_CLASSES,
    CONFIG['num_clients'],
    CONFIG['alpha_dir']
)
client_entropies = [
    compute_client_entropy(cc)
    for cc in client_class_counts
]
client_remaps, client_local_classes = \
    build_label_remap(client_class_counts)

# Preprocess and cache to Drive
DS_CACHE = os.path.join(
    DRIVE_CACHE_DIR, 'OxfordPets_cache'
)
os.makedirs(DS_CACHE, exist_ok=True)

for cid, idxs in enumerate(client_indices):
    if len(idxs) == 0:
        continue
    preprocess_and_save_split(
        raw_train_oxford, idxs, train_labels,
        os.path.join(DS_CACHE, f'client_{cid}'),
        clip_preprocess,
        desc=f"OxfordPets/client_{cid}"
    )

base_test_idx  = [i for i, l in enumerate(test_labels)
                  if l in BASE_CLASSES]
novel_test_idx = [i for i, l in enumerate(test_labels)
                  if l in NOVEL_CLASSES]

preprocess_and_save_split(
    raw_test_oxford, base_test_idx, test_labels,
    os.path.join(DS_CACHE, 'base_test'),
    clip_preprocess, desc="OxfordPets/base_test"
)
preprocess_and_save_split(
    raw_test_oxford, novel_test_idx, test_labels,
    os.path.join(DS_CACHE, 'novel_test'),
    clip_preprocess, desc="OxfordPets/novel_test"
)

# Build DataLoaders
client_loaders = []
for cid in range(CONFIG['num_clients']):
    cache_path = os.path.join(DS_CACHE, f'client_{cid}')
    if not os.path.exists(cache_path) \
            or len(client_indices[cid]) == 0:
        client_loaders.append(None)
        continue
    ds     = DriveTensorDataset(
        cache_path,
        label_remap=client_remaps[cid]
    )
    loader = DataLoader(
        ds,
        batch_size  = CONFIG['batch_size'],
        shuffle     = True,
        num_workers = 0,
        pin_memory  = True,
    )
    client_loaders.append(loader)

base_test_loader = DataLoader(
    DriveTensorDataset(
        os.path.join(DS_CACHE, 'base_test'),
        label_remap=None
    ),
    batch_size=64, shuffle=False, num_workers=0
)
novel_test_loader = DataLoader(
    DriveTensorDataset(
        os.path.join(DS_CACHE, 'novel_test'),
        label_remap=None
    ),
    batch_size=64, shuffle=False, num_workers=0
)

# Per-client text embeddings
# (requires clip_model — loaded in Part 2)
# Defer to after Part 2 runs
# Just confirm variables are set
print(f"\nGlobal variables set:")
print(f"  CLASS_NAMES  : {len(CLASS_NAMES)} classes")
print(f"  BASE_CLASSES : {len(BASE_CLASSES)}")
print(f"  NOVEL_CLASSES: {len(NOVEL_CLASSES)}")
print(f"  client_loaders: "
      f"{sum(1 for l in client_loaders if l)} active")
print(f"  base_test_loader: "
      f"{len(base_test_loader.dataset)} samples")
print(f"  novel_test_loader: "
      f"{len(novel_test_loader.dataset)} samples")

# Free raw datasets
del raw_train_oxford, raw_test_oxford
gc.collect()

print("\nPart 1 complete. Ready for Part 2.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 37.4 MB/s eta 0:00:00
Mounted at /content/drive
Cache directory: /content/drive/MyDrive/blend_oxfordpets_cache
Device: cuda
GPU : Tesla T4
VRAM: 15.6 GB

--- CONFIG ---
  num_clients: 10
  alpha_dir: 0.1
  num_shots: 16
  adapter_rank_vis: 4
  adapter_rank_txt: 4
  proj_rank: 16
  adapter_layers: (1, 12)
  num_rounds: 25
  local_epochs: 1
  local_lr: 0.0003
  batch_size: 8
  temperature: 0.02
  lambda_loss: 0.2
  gamma_static: 0.09
  gamma_min: 0.05
  gamma_max: 0.5
  tau_entropy: 3.0
  tau_divergence: 2.0
  gamma_bias: 1.5
  learn_tau: True
  eval_every: 2
  weight_decay: 0.01
  dropout_rate: 0.1
  label_smoothing: 0.1


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]


CLIP preprocessor loaded.

Part 1 complete.
RAM [Part 1 end]: Process=1538MB  System=2.3/13.6GB

Pre-loading OxfordPets context for Parts 2-4...

Loading OxfordPets...


100%|██████████| 792M/792M [00:29<00:00, 26.9MB/s]
100%|██████████| 19.2M/19.2M [00:01<00:00, 13.0MB/s]


  Train: 3680 samples
  Test : 3669  samples
  Classes: 37
  Split created: 18 base / 19 novel
  [OxfordPets/client_0] Preprocessing 31 images...
    31/31
  [OxfordPets/client_0] Saved torch.Size([31, 3, 224, 224]) (9.3 MB)
  [OxfordPets/client_1] Preprocessing 29 images...
    29/29
  [OxfordPets/client_1] Saved torch.Size([29, 3, 224, 224]) (8.7 MB)
  [OxfordPets/client_2] Preprocessing 15 images...
    15/15
  [OxfordPets/client_2] Saved torch.Size([15, 3, 224, 224]) (4.5 MB)
  [OxfordPets/client_3] Preprocessing 44 images...
    44/44
  [OxfordPets/client_3] Saved torch.Size([44, 3, 224, 224]) (13.2 MB)
  [OxfordPets/client_4] Preprocessing 45 images...
    45/45
  [OxfordPets/client_4] Saved torch.Size([45, 3, 224, 224]) (13.5 MB)
  [OxfordPets/client_5] Preprocessing 4 images...
    4/4
  [OxfordPets/client_5] Saved torch.Size([4, 3, 224, 224]) (1.2 MB)
  [OxfordPets/client_6] Preprocessing 24 images...
    24/24
  [OxfordPets/client_6] Saved torch.Size([24, 3, 224, 224]) (7.2 M

In [2]:
# ============================================================
# PART 2 (FIXED v3): CLIP Backbone + Adapter Modules
# ============================================================
# ADD at very top of Part 2 before any imports
# Forces Python to forget old class definitions

# ============================================================
# PART 2 START: Compute text embeddings
# (requires clip_model which is loaded in this part)
# Run this AFTER the clip_model is created below,
# then these globals will be available for Parts 3-4
# ============================================================
# NOTE: base_text_emb and novel_text_emb are computed
# at the bottom of Part 2 after clip_model is loaded.
# client_text_embs is computed here too.

import importlib
import gc
import torch

# Clear all model-related names from namespace
for name in ['Adapter', 'AdapterInjector', 'BlockAdapterHook',
             'PersonalizedProjectionAdapter', 'BLENDClientModel',
             'BLENDLoss', 'DynamicGammaModule']:
    if name in dir():
        exec(f'del {name}')

torch.cuda.empty_cache()
gc.collect()
print("Namespace cleared — old classes removed")
import open_clip
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy
import gc

# ============================================================
# Load CLIP & inspect dims
# ============================================================

clip_model, _, _ = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='openai'
)
clip_model = clip_model.to(DEVICE)
clip_model.eval()

for param in clip_model.parameters():
    param.requires_grad = False

# ── Inspect ALL relevant shapes before writing any model code ──
print("=== Inspecting CLIP internal shapes ===\n")

shape_log = {}

def make_shape_hook(name):
    def hook(module, inp, output):
        if isinstance(output, torch.Tensor):
            shape_log[name] = tuple(output.shape)
        elif isinstance(output, (tuple, list)):
            shape_log[name] = tuple(output[0].shape)
    return hook

handles = []
handles.append(clip_model.visual.transformer.resblocks[0].register_forward_hook(
    make_shape_hook('vis_block_0')))
handles.append(clip_model.visual.transformer.resblocks[-1].register_forward_hook(
    make_shape_hook('vis_block_last')))
handles.append(clip_model.visual.ln_post.register_forward_hook(
    make_shape_hook('vis_ln_post')))

# Also hook the proj step if it exists as a module
# (in some versions it's a Parameter, not a module)
if hasattr(clip_model.visual, 'proj') and isinstance(
        clip_model.visual.proj, nn.Module):
    handles.append(clip_model.visual.proj.register_forward_hook(
        make_shape_hook('vis_proj_module')))

handles.append(clip_model.transformer.resblocks[0].register_forward_hook(
    make_shape_hook('txt_block_0')))

with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224).to(DEVICE)
    out   = clip_model.encode_image(dummy)
    shape_log['encode_image_out'] = tuple(out.shape)
    tok   = open_clip.tokenize(["a cat", "a dog"]).to(DEVICE)
    clip_model.encode_text(tok)

for h in handles:
    h.remove()

print("Shape inventory:")
for k, v in shape_log.items():
    print(f"  {k:<30} : {v}")

# ── Derive dims safely ──
vis_block_shape  = shape_log['vis_block_last']   # whatever layout CLIP uses
txt_block_shape  = shape_log['txt_block_0']
lnpost_shape     = shape_log['vis_ln_post']

D_EMB = out.shape[-1]   # 512 — final shared embedding

# Vision hidden dim is always the last dim
D_VIS = vis_block_shape[-1]   # 768
D_TXT = txt_block_shape[-1]   # 512

# lnpost output tells us exactly what we'll capture
# It could be (batch, seq, d) or (batch, d)
LNPOST_IS_SEQUENCE = len(lnpost_shape) == 3
# CLS token position — open_clip ViT puts CLS at index 0
CLS_IDX = 0

NUM_VIS_LAYERS = len(clip_model.visual.transformer.resblocks)
NUM_TXT_LAYERS = len(clip_model.transformer.resblocks)

print(f"\nD_VIS                    : {D_VIS}")
print(f"D_TXT                    : {D_TXT}")
print(f"D_EMB                    : {D_EMB}")
print(f"ln_post output shape     : {lnpost_shape}")
print(f"ln_post is sequence      : {LNPOST_IS_SEQUENCE}")
print(f"Vision layers            : {NUM_VIS_LAYERS}")
print(f"Text layers              : {NUM_TXT_LAYERS}")


# ============================================================
# Text embeddings (frozen)
# ============================================================

PROMPT_TEMPLATE = "a photo of a {}, a type of pet."

def get_text_embeddings(class_names, class_ids, device):
    texts  = [
        PROMPT_TEMPLATE.format(class_names[c].replace('_', ' ').lower())
        for c in class_ids
    ]
    tokens = open_clip.tokenize(texts).to(device)
    with torch.no_grad():
        feats = clip_model.encode_text(tokens)
        feats = F.normalize(feats, dim=-1)
    return feats.detach()

base_text_emb  = get_text_embeddings(CLASS_NAMES, BASE_CLASSES,  DEVICE)
novel_text_emb = get_text_embeddings(CLASS_NAMES, NOVEL_CLASSES, DEVICE)

print(f"\nBase  text emb : {base_text_emb.shape}")
print(f"Novel text emb : {novel_text_emb.shape}")


# ============================================================
# Adapter Module
# ============================================================

class Adapter(nn.Module):
    """
    LoRA adapter with dropout for regularization.
    Dropout prevents overfitting on tiny client datasets.
    """
    def __init__(self, d_in, d_out, rank, dropout=0.1):
        super().__init__()
        self.A       = nn.Linear(d_in,  rank,  bias=False)
        self.B       = nn.Linear(rank,  d_out, bias=False)
        self.act     = nn.GELU()
        self.dropout = nn.Dropout(p=dropout)

        nn.init.normal_(self.A.weight, std=0.01)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.B(self.dropout(self.act(self.A(x))))


# Verify init
_test_adapter = Adapter(512, 512, 16)
assert _test_adapter.B.weight.norm().item() == 0.0, "B must be zero at init"
assert _test_adapter.A.weight.norm().item() > 0.0,  "A must be nonzero at init"
print(f"Adapter init: A_norm={_test_adapter.A.weight.norm():.4f}, "
      f"B_norm={_test_adapter.B.weight.norm():.4f} ✓")
del _test_adapter
# ============================================================
# Hook-based Adapter Injection
# ============================================================

class BlockAdapterHook:
    def __init__(self, adapter):
        self.adapter = adapter
        self._handle = None

    def register(self, block):
        self._handle = block.register_forward_hook(self._fn)

    def remove(self):
        if self._handle:
            self._handle.remove()
            self._handle = None

    def _fn(self, module, inp, output):
        # output: (..., d) — adapter handles any leading dims
        return output + self.adapter(output)


class AdapterInjector(nn.Module):
    def __init__(self, blocks, d_model, rank, dropout=0.1):
        super().__init__()
        self.adapters = nn.ModuleList([
            Adapter(d_model, d_model, rank, dropout=dropout)
            for _ in range(len(blocks))
        ])
        self._hooks = []
        for block, adapter in zip(blocks, self.adapters):
            h = BlockAdapterHook(adapter)
            h.register(block)
            self._hooks.append(h)

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()

    def reinstall_hooks(self, blocks):
        self.remove_hooks()
        self._hooks = []
        for block, adapter in zip(blocks, self.adapters):
            h = BlockAdapterHook(adapter)
            h.register(block)
            self._hooks.append(h)


# ============================================================
# Personalized Projection Adapter (P-head)
# ============================================================

class PersonalizedProjectionAdapter(nn.Module):
    def __init__(self, d_vis, rank):
        super().__init__()
        self.adapter = Adapter(d_vis, d_vis, rank)

    def forward(self, cls_token, vis_proj):
        """
        cls_token : (batch, D_VIS)
        vis_proj  : (D_VIS, D_EMB)  — frozen CLIP projection weight
        """
        cls_hat = cls_token + self.adapter(cls_token)   # (batch, D_VIS)
        return cls_hat @ vis_proj                        # (batch, D_EMB)


# ============================================================
# BLEND Client Model
# ============================================================

class BLENDClientModel(nn.Module):
    def __init__(self, clip_model, config):
        super().__init__()
        self.clip   = clip_model
        self.config = config

        # Inside BLENDClientModel.__init__, replace injector construction:
        self.vis_injector = AdapterInjector(
            blocks   = clip_model.visual.transformer.resblocks,
            d_model  = D_VIS,
            rank     = config['adapter_rank_vis'],
            dropout  = config.get('dropout_rate', 0.1),
        )
        self.txt_injector = AdapterInjector(
            blocks   = clip_model.transformer.resblocks,
            d_model  = D_TXT,
            rank     = config['adapter_rank_txt'],
            dropout  = config.get('dropout_rate', 0.1),
        )
        self.proj_adapter = PersonalizedProjectionAdapter(
            d_vis = D_VIS,
            rank  = config['proj_rank'],
        )

        # Capture ln_post output
        # Shape is (batch, seq, D_VIS) as we now know
        self._lnpost_out  = None
        self._lnpost_hook = clip_model.visual.ln_post.register_forward_hook(
            self._capture_lnpost
        )

    def _capture_lnpost(self, module, inp, output):
        # Store full output — we slice CLS in encode_image_adapted
        self._lnpost_out = output
        return output   # do not modify

    def _extract_cls_token(self, lnpost_output, batch_size):
        """
        Extract CLS token from ln_post output regardless of shape.

        Possible shapes:
          (batch, seq, D_VIS)  → batch-first sequence  → CLS at [:, 0, :]
          (seq, batch, D_VIS)  → seq-first sequence    → CLS at [0, :, :]
          (batch, D_VIS)       → already CLS only      → return as-is
        """
        t = lnpost_output
        if t.dim() == 2:
            # Already (batch, D_VIS)
            assert t.shape == (batch_size, D_VIS), \
                f"2D ln_post: expected ({batch_size},{D_VIS}), got {t.shape}"
            return t

        elif t.dim() == 3:
            if t.shape[0] == batch_size:
                # (batch, seq, D_VIS) — batch first
                cls = t[:, CLS_IDX, :]          # (batch, D_VIS)
            else:
                # (seq, batch, D_VIS) — seq first
                cls = t[CLS_IDX, :, :]          # (batch, D_VIS)

            assert cls.shape == (batch_size, D_VIS), \
                f"CLS extraction failed: expected ({batch_size},{D_VIS}), got {cls.shape}"
            return cls

        else:
            raise ValueError(
                f"Unexpected ln_post output dims: {t.shape}"
            )

    def encode_image_adapted(self, images):
        """
        Vision forward with adapters + P-head.
        Returns R_gen (batch, D_EMB), R_per (batch, D_EMB) — both normalized.
        """
        self._lnpost_out = None
        batch_size       = images.shape[0]

        # encode_image runs full ViT — adapter hooks fire inside
        R_gen_raw = self.clip.encode_image(images)   # (batch, D_EMB)
        R_gen     = F.normalize(R_gen_raw, dim=-1)

        assert self._lnpost_out is not None, \
            "ln_post hook never fired — CLIP internals may have changed"

        # Extract CLS token — handles any layout
        cls_token = self._extract_cls_token(self._lnpost_out, batch_size)
        # cls_token: (batch, D_VIS)

        vis_proj  = self.clip.visual.proj             # (D_VIS, D_EMB)
        R_per_raw = self.proj_adapter(cls_token, vis_proj)
        R_per     = F.normalize(R_per_raw, dim=-1)

        return R_gen, R_per

    def forward(self, images, text_embeddings, gamma):
        R_gen, R_per = self.encode_image_adapted(images)
        batch_size   = R_gen.shape[0]

        # ── Gamma shape handling ─────────────────────────────
        if isinstance(gamma, torch.Tensor):
            if gamma.dim() == 0:
                # 0-dim scalar tensor
                gamma = gamma.view(1, 1).expand(batch_size, 1)
            elif gamma.dim() == 1:
                # (batch,) → (batch, 1)
                assert gamma.shape[0] == batch_size, \
                    f"gamma batch {gamma.shape[0]} ≠ image batch {batch_size}"
                gamma = gamma.unsqueeze(-1)
            # else already (batch, 1)
        # float/int: broadcasts with (batch, D_EMB) automatically

        # R^Fus = (1-γ)·R^Gen + γ·R^Per
        R_fus = (1 - gamma) * R_gen + gamma * R_per
        R_fus = F.normalize(R_fus, dim=-1)

        W       = text_embeddings      # (C, D_EMB)
        sim_fus = R_fus @ W.T          # (batch, C)
        sim_anc = R_gen @ W.T          # (batch, C)

        return R_gen, R_per, R_fus, sim_fus, sim_anc

    def get_trainable_params(self):
        return (
            list(self.vis_injector.parameters()) +
            list(self.txt_injector.parameters()) +
            list(self.proj_adapter.parameters())
        )

    def get_aggregatable_params(self):
        return {
            'vis': copy.deepcopy(self.vis_injector.state_dict()),
            'txt': copy.deepcopy(self.txt_injector.state_dict()),
        }

    def set_aggregatable_params(self, p):
        self.vis_injector.load_state_dict(p['vis'])
        self.txt_injector.load_state_dict(p['txt'])

    def get_local_params(self):
        return {'proj': copy.deepcopy(self.proj_adapter.state_dict())}

    def set_local_params(self, p):
        self.proj_adapter.load_state_dict(p['proj'])


# ============================================================
# BLEND Loss
# ============================================================

class BLENDLoss(nn.Module):
    def __init__(self, lambda_loss, temperature, label_smoothing=0.1):
        super().__init__()
        self.lam  = lambda_loss
        self.temp = temperature
        # Label smoothing prevents overconfident predictions
        # critical when training on only 8-48 samples per client
        self.ce   = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    def forward(self, sim_fus, sim_anc, labels):
        L_fus  = self.ce(sim_fus / self.temp, labels)
        L_anch = self.ce(sim_anc / self.temp, labels)
        L      = (1 - self.lam) * L_anch + self.lam * L_fus
        return L, L_fus.item(), L_anch.item()


# ============================================================
# Sanity checks
# ============================================================

print("\n" + "="*55)
print("SANITY CHECK 1: ln_post shape")
print("="*55)
print(f"ln_post output shape from inspection : {lnpost_shape}")
print(f"LNPOST_IS_SEQUENCE                   : {LNPOST_IS_SEQUENCE}")
print(f"CLS extraction will use              : "
      f"{'[:, 0, :]' if lnpost_shape[0]==2 else '[0, :, :]' if len(lnpost_shape)==3 else 'as-is'}")

print("\n" + "="*55)
print("SANITY CHECK 2: Forward pass shapes")
print("="*55)

test_model = BLENDClientModel(clip_model, CONFIG).to(DEVICE)
dummy_imgs = torch.randn(4, 3, 224, 224).to(DEVICE)
test_txt   = base_text_emb[:5]

# Float gamma
R_gen, R_per, R_fus, sim_fus, sim_anc = test_model(
    dummy_imgs, test_txt, gamma=0.09
)
print(f"Float gamma=0.09")
print(f"  R_gen  : {R_gen.shape}  norm={R_gen.norm(dim=-1).mean():.4f}")
print(f"  R_per  : {R_per.shape}  norm={R_per.norm(dim=-1).mean():.4f}")
print(f"  R_fus  : {R_fus.shape}  norm={R_fus.norm(dim=-1).mean():.4f}")
print(f"  sim_fus: {sim_fus.shape}")
print(f"  sim_anc: {sim_anc.shape}")

# Tensor gamma (dynamic mode)
g_tensor = torch.rand(4).to(DEVICE) * 0.2 + 0.05  # random in [0.05, 0.25]
R_g2, R_p2, R_f2, sf2, sa2 = test_model(
    dummy_imgs, test_txt, gamma=g_tensor
)
print(f"\nTensor gamma shape={g_tensor.shape}, values={g_tensor.tolist()}")
print(f"  R_fus2 : {R_f2.shape}  (must match {R_fus.shape})")
assert R_fus.shape == R_f2.shape
print("  Shape check: PASSED ✓")

print("\n" + "="*55)
print("SANITY CHECK 3: Cosine similarity at init")
print("="*55)
cos_val = F.cosine_similarity(R_gen, R_per).mean().item()
print(f"cos(R_gen, R_per) = {cos_val:.6f}  (expect ≈ 1.0)")
assert cos_val > 0.95, f"Too low: {cos_val}"
print("PASSED ✓")

print("\n" + "="*55)
print("SANITY CHECK 4: Loss at init")
print("="*55)
loss_fn   = BLENDLoss(CONFIG['lambda_loss'], CONFIG['temperature'])
lbl_dummy = torch.zeros(4, dtype=torch.long).to(DEVICE)
L, Lf, La = loss_fn(sim_fus, sim_anc, lbl_dummy)
expected  = np.log(5) / CONFIG['temperature']
print(f"Loss      : {L.item():.4f}")
print(f"Expected  : {expected:.4f}  (log(5)/τ)")
print("(Close enough is fine — exact value depends on logit scale)")

print("\n" + "="*55)
print("SANITY CHECK 5: Gradient flow")
print("="*55)
L.backward()

checks = {
    'vis_adapters[0].A' : test_model.vis_injector.adapters[0].A.weight,
    'vis_adapters[0].B' : test_model.vis_injector.adapters[0].B.weight,
    'vis_adapters[-1].A': test_model.vis_injector.adapters[-1].A.weight,
    # 'txt_adapters[0].A' : test_model.txt_injector.adapters[0].A.weight, # This param is not exercised by the current forward pass.
    'proj_adapter.A'    : test_model.proj_adapter.adapter.A.weight,
    'proj_adapter.B'    : test_model.proj_adapter.adapter.B.weight,
}
all_ok = True
for name, param in checks.items():
    has_grad = param.grad is not None
    norm     = param.grad.norm().item() if has_grad else 0.0
    ok       = has_grad   # B grad=0 at init is expected and fine, but grad must not be None.
    mark     = '✓' if ok else '✗'
    print(f"  {mark}  {name:<28}  grad_norm={norm:.8f}")
    if not ok:
        all_ok = False

assert all_ok, "Missing gradients — do not proceed"
print("\nAll gradient checks: PASSED ✓")

print("\n" + "="*55)
print("SANITY CHECK 6: Parameter count")
print("="*55)
vis_p  = sum(p.numel() for p in test_model.vis_injector.parameters())
txt_p  = sum(p.numel() for p in test_model.txt_injector.parameters())
proj_p = sum(p.numel() for p in test_model.proj_adapter.parameters())
print(f"  Vision adapters (agg)  : {vis_p:>10,}")
print(f"  Text adapters   (agg)  : {txt_p:>10,}")
print(f"  Proj adapter    (local): {proj_p:>10,}")
print(f"  Total trainable        : {vis_p+txt_p+proj_p:>10,}")



del test_model
torch.cuda.empty_cache()
gc.collect()
print("\nPart 2 complete.")

# -- Compute text embeddings now that clip_model exists ------
print("\nComputing text embeddings for OxfordPets...")

base_text_emb  = get_text_embeddings(
    CLASS_NAMES, BASE_CLASSES,  DEVICE
)
novel_text_emb = get_text_embeddings(
    CLASS_NAMES, NOVEL_CLASSES, DEVICE
)

client_text_embs = []
for cid, counts in enumerate(client_class_counts):
    present = sorted(
        [c for c, n in counts.items() if n > 0]
    )
    if not present:
        client_text_embs.append(None)
        continue
    emb = get_text_embeddings(
        CLASS_NAMES, present, DEVICE
    )
    client_text_embs.append(emb)

print(f"base_text_emb  : {base_text_emb.shape}")
print(f"novel_text_emb : {novel_text_emb.shape}")
print(f"client_text_embs: "
      f"{sum(1 for e in client_text_embs if e is not None)} "
      f"clients ready")

print("\nParts 1+2 complete. Ready for Parts 3, 4, 5.")

Namespace cleared — old classes removed
=== Inspecting CLIP internal shapes ===

Shape inventory:
  vis_block_0                    : (2, 50, 768)
  vis_block_last                 : (2, 50, 768)
  vis_ln_post                    : (2, 50, 768)
  encode_image_out               : (2, 512)
  txt_block_0                    : (2, 77, 512)

D_VIS                    : 768
D_TXT                    : 512
D_EMB                    : 512
ln_post output shape     : (2, 50, 768)
ln_post is sequence      : True
Vision layers            : 12
Text layers              : 12

Base  text emb : torch.Size([18, 512])
Novel text emb : torch.Size([19, 512])
Adapter init: A_norm=0.9242, B_norm=0.0000 ✓

SANITY CHECK 1: ln_post shape
ln_post output shape from inspection : (2, 50, 768)
LNPOST_IS_SEQUENCE                   : True
CLS extraction will use              : [:, 0, :]

SANITY CHECK 2: Forward pass shapes
Float gamma=0.09
  R_gen  : torch.Size([4, 512])  norm=1.0000
  R_per  : torch.Size([4, 512])  norm=1.0

In [3]:
# ============================================================
# PART 3 (FIXED): Dynamic Gamma Module
# Same math, added detailed debug checks
# ============================================================

class DynamicGammaModule(nn.Module):
    """
    γ_n^(k) = clamp(σ(τ_e·(1-H̃_n) + τ_d·div(R_gen,R_per) - c), γ_min, γ_max)

    τ_e, τ_d passed through softplus to ensure positivity.
    Initialized so that:
      - skewed client  (H̃≈0), untrained P-head (cos≈1): γ ≈ 0.23
      - uniform client (H̃≈1), untrained P-head (cos≈1): γ ≈ 0.18
    Both above paper's 0.09 initially, but they learn down/up from there.
    """

    def __init__(self, config):
        super().__init__()
        self.gamma_min = config['gamma_min']
        self.gamma_max = config['gamma_max']

        if config['learn_tau']:
            self.tau_e = nn.Parameter(
                torch.tensor(config['tau_entropy'],   dtype=torch.float32))
            self.tau_d = nn.Parameter(
                torch.tensor(config['tau_divergence'], dtype=torch.float32))
            self.bias  = nn.Parameter(
                torch.tensor(config['gamma_bias'],    dtype=torch.float32))
        else:
            self.register_buffer('tau_e',
                torch.tensor(config['tau_entropy']))
            self.register_buffer('tau_d',
                torch.tensor(config['tau_divergence']))
            self.register_buffer('bias',
                torch.tensor(config['gamma_bias']))

    def forward(self, R_gen, R_per, client_entropy):
        """
        R_gen, R_per   : (batch, D_EMB) — already normalized
        client_entropy : float ∈ [0,1]
        Returns: gamma (batch,), metrics dict
        """
        tau_e = F.softplus(self.tau_e)
        tau_d = F.softplus(self.tau_d)

        # Entropy term: scalar, same for all samples in this client's batch
        entropy_term = torch.tensor(
            1.0 - client_entropy,
            dtype=torch.float32,
            device=R_gen.device
        )

        # Divergence term: per-sample
        cos_sim        = F.cosine_similarity(R_gen.detach(),
                                             R_per.detach(), dim=-1)
        divergence     = (1.0 - cos_sim) / 2.0   # normalize to [0,1]

        logit = tau_e * entropy_term + tau_d * divergence - self.bias
        gamma = torch.sigmoid(logit)
        gamma = torch.clamp(gamma, self.gamma_min, self.gamma_max)

        metrics = {
            'gamma_mean'      : gamma.mean().item(),
            'gamma_std'       : gamma.std().item() if gamma.numel() > 1 else 0.0,
            'entropy_term'    : entropy_term.item(),
            'cos_sim_mean'    : cos_sim.mean().item(),
            'divergence_mean' : divergence.mean().item(),
            'tau_e'           : tau_e.item(),
            'tau_d'           : tau_d.item(),
            'bias'            : self.bias.item(),
        }

        return gamma, metrics


# ============================================================
# Sanity check
# ============================================================

print("=== Dynamic Gamma Sanity Check ===\n")
test_dyn = DynamicGammaModule(CONFIG).to(DEVICE)

cases = [
    ("Skewed  (H̃=0.05), untrained (cos≈1.00)", 0.05, 1.00),
    ("Skewed  (H̃=0.05), trained   (cos=0.50)",  0.05, 0.50),
    ("Uniform (H̃=0.95), untrained (cos≈1.00)", 0.95, 1.00),
    ("Uniform (H̃=0.95), trained   (cos=0.50)", 0.95, 0.50),
]

print(f"{'Scenario':<48} {'γ':>7}  {'τ_e·(1-H̃)':>10}  {'τ_d·div':>8}")
print("-" * 80)

with torch.no_grad():
    for desc, H, cos_val in cases:
        batch = 8
        r_g = F.normalize(torch.randn(batch, D_EMB).to(DEVICE), dim=-1)
        noise = F.normalize(torch.randn(batch, D_EMB).to(DEVICE), dim=-1)
        r_p = F.normalize(cos_val * r_g + (1 - cos_val) * noise, dim=-1)

        g, m = test_dyn(r_g, r_p, H)

        tau_e_val = m['tau_e'] * (1.0 - H)
        tau_d_val = m['tau_d'] * m['divergence_mean']

        print(f"{desc:<48} {m['gamma_mean']:>7.4f}  "
              f"{tau_e_val:>10.4f}  {tau_d_val:>8.4f}")

del test_dyn
torch.cuda.empty_cache()
print("\nPart 3 complete.")

=== Dynamic Gamma Sanity Check ===

Scenario                                               γ  τ_e·(1-H̃)   τ_d·div
--------------------------------------------------------------------------------
Skewed  (H̃=0.05), untrained (cos≈1.00)           0.5000      2.8962   -0.0000
Skewed  (H̃=0.05), trained   (cos=0.50)           0.5000      2.8962    0.3128
Uniform (H̃=0.95), untrained (cos≈1.00)           0.2063      0.1524   -0.0000
Uniform (H̃=0.95), trained   (cos=0.50)           0.2620      0.1524    0.3118

Part 3 complete.


In [4]:
import copy
import time
import numpy as np
from torch.optim import SGD
from torch.optim.lr_scheduler import LinearLR


# ============================================================
# Per-client text embeddings (correct class subset)
# ============================================================

print("Computing per-client text embeddings...")
client_text_embs      = []
client_local_classes  = []
client_remaps         = []

for cid, counts in enumerate(client_class_counts):
    # Only classes with actual samples
    present = sorted([c for c, n in counts.items() if n > 0])

    if len(present) == 0:
        client_text_embs.append(None)
        client_local_classes.append([])
        client_remaps.append({})
        continue

    # Global class id → local index [0, len(present))
    remap = {c: i for i, c in enumerate(present)}

    # Text embedding for this client's class subset
    emb = get_text_embeddings(CLASS_NAMES, present, DEVICE)

    client_text_embs.append(emb)
    client_local_classes.append(present)
    client_remaps.append(remap)

    print(f"  Client {cid:2d}: {len(present):2d} classes, "
          f"text emb: {emb.shape}, "
          f"remap sample: {list(remap.items())[:3]}")


# ============================================================
# Label remapping utility
# ============================================================

def remap_labels(labels_tensor, remap_dict, device):
    """
    Map global class IDs to local indices.
    Any label not in remap gets -1 (filtered out by loss if needed).
    """
    return torch.tensor(
        [remap_dict.get(l.item(), -1) for l in labels_tensor],
        dtype=torch.long, device=device
    )


# ============================================================
# Gradient flow verification
# Run this before training to confirm gradients reach adapters
# ============================================================

def verify_gradient_flow(config, clip_model, device):
    print("\n=== Gradient Flow Verification ===")

    model   = BLENDClientModel(clip_model, config).to(device)
    loss_fn = BLENDLoss(config['lambda_loss'], config['temperature'])

    # Use 2 classes for simplicity
    test_classes = BASE_CLASSES[:2]
    test_text    = get_text_embeddings(CLASS_NAMES, test_classes, device)
    test_imgs    = torch.randn(4, 3, 224, 224).to(device)
    test_labels  = torch.zeros(4, dtype=torch.long).to(device)

    # Forward
    _, _, _, sim_fus, sim_anc = model(test_imgs, test_text, gamma=0.09)
    loss, _, _ = loss_fn(sim_fus, sim_anc, test_labels)
    loss.backward()

    print(f"Loss value: {loss.item():.4f}")
    # Corrected: CrossEntropyLoss expects logits, which are `sim / temp`. If `sim` is near 0, then `sim/temp` is near 0.
    # In this case, `CE(zeros)` is `log(num_classes)`. Here num_classes is 2.
    print(f"Expected (~log(2)): {np.log(2):.4f}")

    # Check gradients
    checks = {
        'vis_adapter[0].A' : model.vis_injector.adapters[0].A.weight,
        'vis_adapter[0].B' : model.vis_injector.adapters[0].B.weight,
        'vis_adapter[-1].A': model.vis_injector.adapters[-1].A.weight,
        # 'txt_adapter[0].A' : model.txt_injector.adapters[0].A.weight, # This param is not exercised by the current forward pass.
        'proj_adapter.A'   : model.proj_adapter.adapter.A.weight,
        'proj_adapter.B'   : model.proj_adapter.adapter.B.weight,
    }

    all_ok = True
    for name, param in checks.items():
        if param.grad is None:
            print(f"  ✘ {name:<25} MISSING GRADIENT")
            all_ok = False
        else:
            print(f"  ✓ {name:<25} grad_norm={param.grad.norm():.6f}")

    if all_ok:
        print("\nAll gradients flowing correctly ✓")
    else:
        print("\nWARNING: Some gradients missing ✘")

    del model, test_imgs, sim_fus, sim_anc, loss
    torch.cuda.empty_cache()
    gc.collect()
    return all_ok

gradient_ok = verify_gradient_flow(CONFIG, clip_model, DEVICE)


# ============================================================
# COMPLETE FIXED train_client_one_round
# ============================================================

def train_client_one_round(
    model,
    loader,
    model_optimizer,
    scheduler,
    loss_fn,
    text_emb,
    label_remap,
    device,
    dyn_gamma=None,
    dyn_gamma_optimizer=None,
    client_entropy=None,
    static_gamma=None,
):
    model.train()
    if dyn_gamma is not None:
        dyn_gamma.train()

    total_loss = 0.0
    n_batches  = 0
    gamma_log  = []
    n_correct  = 0
    n_total    = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        valid  = labels >= 0
        if valid.sum() == 0:
            continue
        images = images[valid]
        labels = labels[valid]

        model_optimizer.zero_grad()
        # Do NOT zero dyn_gamma grad here —
        # managed at round level in run_federated_training

        if dyn_gamma is not None:
            # ── Step 1: Get embeddings (detached from model graph)
            # We detach here to avoid computing gradients
            # through the full CLIP encoder twice.
            # This is correct because tau params do NOT
            # depend on R_gen/R_per — they only depend
            # on the SCALAR cosine similarity between them.
            with torch.no_grad():
                R_gen_det, R_per_det = \
                    model.encode_image_adapted(images)

            # ── Step 2: Compute cosine divergence
            # This is a scalar computation — no heavy gradients
            cos_sim    = F.cosine_similarity(
                R_gen_det, R_per_det, dim=-1
            )  # (batch,) — pure scalar, detached from encoder

            divergence = ((1.0 - cos_sim) / 2.0).detach()
            # divergence is now a plain tensor, not connected
            # to the CLIP encoder weights

            # ── Step 3: Compute gamma WITH gradient to tau
            # This is the key: gamma depends on tau_e, tau_d, c
            # AND on divergence (which is a plain tensor)
            # So gradient flows: loss → gamma → tau ✓
            # But NOT: loss → gamma → CLIP encoder (too expensive)

            tau_e = F.softplus(dyn_gamma.tau_e)
            tau_d = F.softplus(dyn_gamma.tau_d)

            entropy_term = torch.tensor(
                1.0 - client_entropy,
                dtype=torch.float32,
                device=device
            )  # scalar, no grad needed

            # logit depends on tau_e, tau_d, c (all have grad)
            # and on divergence (detached scalar — that is fine)
            logit = (
                tau_e * entropy_term    # grad flows to tau_e ✓
                + tau_d * divergence    # grad flows to tau_d ✓
                - dyn_gamma.bias        # grad flows to bias  ✓
            )  # (batch,)

            gamma = torch.sigmoid(logit)
            gamma = torch.clamp(
                gamma,
                dyn_gamma.gamma_min,
                dyn_gamma.gamma_max
            )
            # gamma is now connected to tau_e, tau_d, bias
            # through logit — gradient will flow ✓

            gm = {
                'gamma_mean'    : gamma.mean().item(),
                'gamma_std'     : gamma.std().item()
                                  if gamma.numel()>1 else 0.0,
                'entropy_term'  : entropy_term.item(),
                'cos_sim_mean'  : cos_sim.mean().item(),
                'divergence_mean': divergence.mean().item(),
                'tau_e'         : tau_e.item(),
                'tau_d'         : tau_d.item(),
                'bias'          : dyn_gamma.bias.item(),
            }
            gamma_log.append(gm)

        else:
            gamma = static_gamma

        # ── Step 4: Full forward pass
        # gamma (if dynamic) is connected to tau params
        # loss.backward() will flow through gamma to tau ✓
        R_gen, R_per, R_fus, sim_fus, sim_anc = model(
            images, text_emb, gamma=gamma
        )

        loss, lf, la = loss_fn(sim_fus, sim_anc, labels)

        if torch.isnan(loss) or torch.isinf(loss):
            print(f"  WARNING: NaN/Inf loss — skipping")
            continue

        # ── Step 5: Backward
        # Gradients flow to:
        #   - model adapter params (via R_gen, R_per, R_fus)
        #   - tau_e, tau_d, bias (via gamma → logit)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=0.5
        )
        if dyn_gamma is not None:
            torch.nn.utils.clip_grad_norm_(
                dyn_gamma.parameters(), max_norm=0.1
            )

        model_optimizer.step()

        total_loss += loss.item()
        n_batches  += 1

        with torch.no_grad():
            preds     = sim_fus.argmax(dim=-1)
            n_correct += (preds == labels).sum().item()
            n_total   += len(labels)

    avg_loss  = total_loss / max(n_batches, 1)
    train_acc = n_correct / max(n_total, 1) * 100
    if gamma_log:
        avg_gamma = np.mean([m['gamma_mean'] for m in gamma_log])
    elif static_gamma is not None:
        avg_gamma = float(static_gamma)
    else:
        # Dynamic mode but empty loader — compute gamma from entropy alone
        # sigmoid(tau_e * (1-H) - bias) as fallback
        avg_gamma = float(CONFIG['gamma_min'])

    return avg_loss, train_acc, gamma_log, avg_gamma


# ============================================================
# FedAvg aggregation
# ============================================================

def fedavg_aggregate(client_models, client_sizes):
    total = sum(client_sizes)
    if total == 0:
        return None

    agg = None
    for model, size in zip(client_models, client_sizes):
        if size == 0:
            continue
        w      = size / total
        params = model.get_aggregatable_params()

        if agg is None:
            agg = {
                k: {pk: pv.clone() * w for pk, pv in sd.items()}
                for k, sd in params.items()
            }
        else:
            for k, sd in params.items():
                for pk, pv in sd.items():
                    agg[k][pk] += pv * w

    return agg


# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    base_loader,
    novel_loader,
    base_text_emb,
    novel_text_emb,
    base_classes,
    novel_classes,
    device,
    dyn_gamma=None,
    client_entropy=None,
    static_gamma=None,
):
    model.eval()

    def run_split(loader, text_emb, class_ids):
        # Build eval remap: global id → local eval index
        eval_remap = {c: i for i, c in enumerate(class_ids)}

        correct = 0
        total   = 0

        for images, global_labels in loader:
            images        = images.to(device, non_blocking=True)
            global_labels = global_labels.to(device)

            # Remap global labels to eval-local indices
            local_labels = remap_labels(global_labels, eval_remap, device)

            # Filter any labels not in this split
            valid = local_labels >= 0
            if valid.sum() == 0:
                continue
            images       = images[valid]
            local_labels = local_labels[valid]

            # Compute gamma
            if dyn_gamma is not None:
                R_gen, R_per = model.encode_image_adapted(images)
                gamma, _     = dyn_gamma(R_gen, R_per,
                                         client_entropy or 0.5)
            else:
                gamma = static_gamma

            _, _, R_fus, _, _ = model(images, text_emb, gamma=gamma)

            # Predict: argmax of cosine similarity
            sim   = R_fus @ text_emb.T   # (batch, num_eval_classes)
            preds = sim.argmax(dim=-1)

            correct += (preds == local_labels).sum().item()
            total   += local_labels.numel()

        return (correct / total * 100) if total > 0 else 0.0

    per_acc = run_split(base_loader,  base_text_emb,  base_classes)
    gen_acc = run_split(novel_loader, novel_text_emb, novel_classes)
    h_mean  = (2 * per_acc * gen_acc / (per_acc + gen_acc)
               if (per_acc + gen_acc) > 0 else 0.0)

    return per_acc, gen_acc, h_mean


# ============================================================
# Main training function
# ============================================================

# The problem in the training loop:
# dyn_gamma parameters were added to optimizer but
# the optimizer is rebuilt EVERY round inside the client loop
# So dyn_gamma params get a fresh optimizer each round
# with lr warmup resetting them — they never accumulate momentum

# Fix: build ONE optimizer for dyn_gamma OUTSIDE the client loop
# and a separate optimizer per client for model params

def run_federated_training(mode, config):
    assert mode in ('static', 'dynamic')
    print(f"\n{'='*60}")
    print(f"  Mode: {mode.upper()} GAMMA BLEND")
    print(f"{'='*60}\n")

    client_models = [
        BLENDClientModel(clip_model, config).to(DEVICE)
        for _ in range(config['num_clients'])
    ]

    dyn_gamma = DynamicGammaModule(config).to(DEVICE) \
                if mode == 'dynamic' else None

    dyn_gamma_optimizer = None
    if dyn_gamma is not None:
        dyn_gamma_optimizer = torch.optim.Adam(
            dyn_gamma.parameters(),
            lr    = 1e-3,
            betas = (0.9, 0.999),
        )

    loss_fn = BLENDLoss(
        config['lambda_loss'],
        config['temperature'],
        label_smoothing=config.get('label_smoothing', 0.1)
    )

    history = {
        'round'            : [],
        'per_acc'          : [],
        'gen_acc'          : [],
        'h_mean'           : [],
        'train_loss'       : [],
        'train_acc'        : [],
        'gamma_mean_round' : [],
        'gamma_per_client' : [[] for _ in range(config['num_clients'])],
        'tau_e_history'    : [],
        'tau_d_history'    : [],
        'bias_history'     : [],
    }

    best_h = 0.0
    t0     = time.time()
    sizes  = [len(idxs) for idxs in client_indices]

    for rnd in range(1, config['num_rounds'] + 1):

        round_losses = []
        round_accs   = []
        round_gammas = []

        # ── Zero dyn_gamma grad at START of round ────────────
        # Must happen BEFORE client loop so grads accumulate
        # across all clients, then we step AFTER client loop
        if dyn_gamma_optimizer is not None:
            dyn_gamma_optimizer.zero_grad()

        # ── Local Training ───────────────────────────────────
        for cid in range(config['num_clients']):
            loader   = client_loaders[cid]
            text_emb = client_text_embs[cid]
            remap    = client_remaps[cid]
            entropy  = client_entropies[cid]

            if loader is None or text_emb is None \
                    or len(remap) == 0:
                continue

            model = client_models[cid]

            model_optimizer = torch.optim.AdamW(
                model.get_trainable_params(),
                lr           = config['local_lr'],
                betas        = (0.9, 0.999),
                weight_decay = config.get('weight_decay', 0.01),
                eps          = 1e-8,
            )

            avg_loss, train_acc, gamma_log, avg_gamma = \
                train_client_one_round(
                    model               = model,
                    loader              = loader,
                    model_optimizer     = model_optimizer,
                    scheduler           = None,
                    loss_fn             = loss_fn,
                    text_emb            = text_emb,
                    label_remap         = remap,
                    device              = DEVICE,
                    dyn_gamma           = dyn_gamma,
                    dyn_gamma_optimizer = None,
                    client_entropy      = entropy,
                    static_gamma        = config['gamma_static'] \
                                          if mode == 'static' \
                                          else None,
                )

            round_losses.append(avg_loss)
            round_accs.append(train_acc)
            round_gammas.append(avg_gamma)
            history['gamma_per_client'][cid].append(avg_gamma)

        # ── Tau gradient check BEFORE step (round 1 only) ────
        # MUST be here — after client loop, before step()
        # step() zeros grads so checking after step = always None
        if mode == 'dynamic' and rnd == 1 \
                and dyn_gamma is not None:
            print("\n--- Round 1 Tau Gradient Check ---")
            all_ok = True
            for name, param in [
                ('tau_e', dyn_gamma.tau_e),
                ('tau_d', dyn_gamma.tau_d),
                ('bias',  dyn_gamma.bias),
            ]:
                if param.grad is None:
                    print(f"  {name:<8}: None ✗")
                    all_ok = False
                else:
                    print(f"  {name:<8}: "
                          f"{param.grad.item():.10f} ✓")
            if all_ok:
                print("  All tau grads flowing ✓")
            else:
                print("  WARNING: missing grads ✗")
                print("  → check train_client_one_round fix")
            print("----------------------------------\n")

        # ── Step dyn_gamma AFTER grad check ──────────────────
        if dyn_gamma_optimizer is not None:
            dyn_gamma_optimizer.step()
            # NO zero_grad here — done at top of next round

            tau_e = F.softplus(dyn_gamma.tau_e).item()
            tau_d = F.softplus(dyn_gamma.tau_d).item()
            bias  = dyn_gamma.bias.item()
            history['tau_e_history'].append(tau_e)
            history['tau_d_history'].append(tau_d)
            history['bias_history'].append(bias)

        # ── FedAvg Aggregation ───────────────────────────────
        local_backups = [m.get_local_params()
                         for m in client_models]
        agg_params    = fedavg_aggregate(client_models, sizes)
        if agg_params is not None:
            for m, lp in zip(client_models, local_backups):
                m.set_aggregatable_params(agg_params)
                m.set_local_params(lp)

        # ── Logging ──────────────────────────────────────────
        avg_loss  = np.mean(round_losses) if round_losses else 0
        avg_acc   = np.mean(round_accs)   if round_accs   else 0
        avg_gamma = np.mean(round_gammas) if round_gammas else 0

        history['train_loss'].append(avg_loss)
        history['train_acc'].append(avg_acc)
        history['gamma_mean_round'].append(avg_gamma)

        # ── Evaluation ───────────────────────────────────────
        do_eval = (
            rnd % config['eval_every'] == 0
            or rnd == 1
            or rnd == config['num_rounds']
        )

        if do_eval:
            all_per, all_gen, all_hm = [], [], []

            for cid in range(config['num_clients']):
                if client_loaders[cid] is None:
                    continue
                per, gen, hm = evaluate(
                    model          = client_models[cid],
                    base_loader    = base_test_loader,
                    novel_loader   = novel_test_loader,
                    base_text_emb  = base_text_emb,
                    novel_text_emb = novel_text_emb,
                    base_classes   = BASE_CLASSES,
                    novel_classes  = NOVEL_CLASSES,
                    device         = DEVICE,
                    dyn_gamma      = dyn_gamma,
                    client_entropy = client_entropies[cid],
                    static_gamma   = config['gamma_static'] \
                                     if mode == 'static' \
                                     else None,
                )
                all_per.append(per)
                all_gen.append(gen)
                all_hm.append(hm)

            mean_per = np.mean(all_per)
            mean_gen = np.mean(all_gen)
            mean_hm  = np.mean(all_hm)

            history['round'].append(rnd)
            history['per_acc'].append(mean_per)
            history['gen_acc'].append(mean_gen)
            history['h_mean'].append(mean_hm)

            if mean_hm > best_h:
                best_h = mean_hm

            elapsed = time.time() - t0

            tau_str = ""
            if dyn_gamma is not None \
                    and history['tau_e_history']:
                tau_str = (
                    f" | τ_e={history['tau_e_history'][-1]:.4f}"
                    f" τ_d={history['tau_d_history'][-1]:.4f}"
                    f" b={history['bias_history'][-1]:.4f}"
                )

            print(
                f"  Round {rnd:3d}/{config['num_rounds']} | "
                f"Loss: {avg_loss:.4f} | "
                f"γ: {avg_gamma:.4f} | "
                f"Per: {mean_per:.2f}% | "
                f"Gen: {mean_gen:.2f}% | "
                f"H: {mean_hm:.2f}%"
                f"{tau_str} | [{elapsed:.0f}s]"
            )

    print(f"\nBest H-Mean ({mode}): {best_h:.2f}%")
    return history, client_models, dyn_gamma

# ============================================================
# MODEL SAVE / LOAD UTILITIES
# Add at end of Part 4, before running Part 5
# ============================================================

SAVE_DIR = os.path.join(DRIVE_CACHE_DIR, 'saved_models')
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Model save directory: {SAVE_DIR}")


def save_experiment(
    mode,
    history,
    client_models,
    dyn_gamma=None,
    config=CONFIG,
):
    """
    Save everything needed to resume or evaluate later.
    Saves per-client model params + history + config.
    """
    save_path = os.path.join(SAVE_DIR, f'blend_{mode}.pt')

    # Collect all client states
    client_states = []
    for model in client_models:
        client_states.append({
            'agg'  : model.get_aggregatable_params(),
            'local': model.get_local_params(),
        })

    payload = {
        'mode'          : mode,
        'config'        : config,
        'history'       : history,
        'client_states' : client_states,
        'base_classes'  : BASE_CLASSES,
        'novel_classes' : NOVEL_CLASSES,
        'class_names'   : CLASS_NAMES,
    }

    # Save dynamic gamma module if present
    if dyn_gamma is not None:
        payload['dyn_gamma_state'] = dyn_gamma.state_dict()

    torch.save(payload, save_path)

    size_mb = os.path.getsize(save_path) / 1e6
    print(f"Saved [{mode}] → {save_path}  ({size_mb:.1f} MB)")
    return save_path


def load_experiment(mode):
    """
    Load saved experiment.
    Returns history, client_models, dyn_gamma (or None).
    Prints final metrics immediately so you know what was saved.
    """
    save_path = os.path.join(SAVE_DIR, f'blend_{mode}.pt')

    if not os.path.exists(save_path):
        print(f"No saved model found at {save_path}")
        return None, None, None

    print(f"Loading [{mode}] from {save_path} ...")
    # Fix: Set weights_only=False to allow loading non-tensor objects
    payload = torch.load(save_path, map_location=DEVICE, weights_only=False)

    config        = payload['config']
    history       = payload['history']
    client_states = payload['client_states']

    # Rebuild models
    client_models = [
        BLENDClientModel(clip_model, config).to(DEVICE)
        for _ in range(config['num_clients'])
    ]

    # Restore weights
    for model, state in zip(client_models, client_states):
        model.set_aggregatable_params(state['agg'])
        model.set_local_params(state['local'])

    # Restore dynamic gamma if saved
    dyn_gamma = None
    if 'dyn_gamma_state' in payload:
        dyn_gamma = DynamicGammaModule(config).to(DEVICE)
        dyn_gamma.load_state_dict(payload['dyn_gamma_state'])
        print(f"  Dynamic gamma restored.")
        tau_e = F.softplus(dyn_gamma.tau_e).item()
        tau_d = F.softplus(dyn_gamma.tau_d).item()
        bias  = dyn_gamma.bias.item()
        print(f"  τ_e={tau_e:.4f}  τ_d={tau_d:.4f}  bias={bias:.4f}")

    # Print saved metrics
    if history['round']:
        last   = -1
        best_h = max(history['h_mean'])
        print(f"\n  Restored history: {len(history['round'])} eval points")
        print(f"  Final  → Per:{history['per_acc'][last]:.2f}%  "
              f"Gen:{history['gen_acc'][last]:.2f}%  "
              f"H:{history['h_mean'][last]:.2f}%")
        print(f"  Best H-Mean: {best_h:.2f}%")

    return history, client_models, dyn_gamma


def check_saved_models():
    """Quick check of what's already saved in Drive."""
    print(f"\nSaved models in {SAVE_DIR}:")
    if not os.path.exists(SAVE_DIR):
        print("  Directory does not exist yet.")
        return
    files = [f for f in os.listdir(SAVE_DIR) if f.endswith('.pt')]
    if not files:
        print("  No saved models found.")
        return
    for f in sorted(files):
        full   = os.path.join(SAVE_DIR, f)
        size   = os.path.getsize(full) / 1e6
        # Peek at history without loading full model
        try:
            # Fix: Set weights_only=False to allow loading non-tensor objects
            peek = torch.load(full, map_location='cpu', weights_only=False)
            mode = peek.get('mode', '?')
            rds  = peek['history']['round']
            best = max(peek['history']['h_mean']) if peek['history']['h_mean'] else 0
            print(f"  {f:<30} {size:.1f} MB | "
                  f"mode={mode} | "
                  f"rounds={rds[-1] if rds else 0} | "
                  f"best_H={best:.2f}%")
        except Exception as e:
            print(f"  {f:<30} {size:.1f} MB | (could not read: {e})")

check_saved_models()
print("\nSave/load utilities ready.")

Computing per-client text embeddings...
  Client  0:  4 classes, text emb: torch.Size([4, 512]), remap sample: [(11, 0), (16, 1), (31, 2)]
  Client  1:  5 classes, text emb: torch.Size([5, 512]), remap sample: [(9, 0), (10, 1), (22, 2)]
  Client  2:  3 classes, text emb: torch.Size([3, 512]), remap sample: [(28, 0), (31, 1), (33, 2)]
  Client  3:  6 classes, text emb: torch.Size([6, 512]), remap sample: [(4, 0), (5, 1), (20, 2)]
  Client  4:  4 classes, text emb: torch.Size([4, 512]), remap sample: [(9, 0), (12, 1), (27, 2)]
  Client  5:  1 classes, text emb: torch.Size([1, 512]), remap sample: [(26, 0)]
  Client  6:  3 classes, text emb: torch.Size([3, 512]), remap sample: [(9, 0), (19, 1), (32, 2)]
  Client  7:  2 classes, text emb: torch.Size([2, 512]), remap sample: [(20, 0), (22, 1)]
  Client  8:  6 classes, text emb: torch.Size([6, 512]), remap sample: [(16, 0), (19, 1), (26, 2)]
  Client  9: 18 classes, text emb: torch.Size([18, 512]), remap sample: [(4, 0), (5, 1), (9, 2)]

===

In [5]:
# ============================================================
# CORRECTED Quick Adapter Learning Test
# ============================================================

print("=== Quick Adapter Learning Test ===\n")

# Re-run Part 2 first if B_norm was not 0 above
# Then run this test

test_m   = BLENDClientModel(clip_model, CONFIG).to(DEVICE)
test_opt = torch.optim.Adam(
    test_m.get_trainable_params(),
    lr=CONFIG['local_lr'],
)
test_loss_fn = BLENDLoss(CONFIG['lambda_loss'], CONFIG['temperature'])

# Confirm B starts at zero
b_init = test_m.vis_injector.adapters[0].B.weight.norm().item()
a_init = test_m.vis_injector.adapters[0].A.weight.norm().item()
print(f"Before training:")
print(f"  A norm = {a_init:.6f}  (should be ~0.08)")
print(f"  B norm = {b_init:.6f}  (should be 0.000000)")
assert b_init < 1e-6, f"B not zero! Got {b_init}. Re-run Part 2 first."

# Use client 9 with a FRESH loader (don't consume the training loader)
test_cid = 9
test_emb = client_text_embs[test_cid]

# Build fresh loader from Drive cache
from torch.utils.data import DataLoader
_cache = os.path.join(DRIVE_CACHE_DIR, f'client_{test_cid}')
_ds    = DriveTensorDataset(_cache, label_remap=client_remaps[test_cid])
_fresh_loader = DataLoader(
    _ds,
    batch_size = CONFIG['batch_size'],
    shuffle    = True,
    num_workers= 0,
)

print(f"\nClient {test_cid}: {len(_ds)} samples, "
      f"{len(client_local_classes[test_cid])} classes, "
      f"{len(_fresh_loader)} batches/epoch\n")

print(f"{'Step':<6} {'Loss':>8} {'Acc':>8} "
      f"{'A_norm':>10} {'B_norm':>10} {'Status'}")
print("-" * 60)

prev_loss = None
for step, (imgs, lbls) in enumerate(_fresh_loader):
    if step >= 15:
        break

    imgs  = imgs.to(DEVICE)
    lbls  = lbls.to(DEVICE)
    valid = lbls >= 0
    if valid.sum() == 0:
        continue
    imgs, lbls = imgs[valid], lbls[valid]

    test_opt.zero_grad()

    _, _, _, sim_fus, sim_anc = test_m(imgs, test_emb, gamma=0.09)
    L, _, _ = test_loss_fn(sim_fus, sim_anc, lbls)

    if torch.isnan(L):
        print(f"{step+1:<6} {'NaN':>8} — stopping")
        break

    L.backward()
    torch.nn.utils.clip_grad_norm_(test_m.parameters(), max_norm=0.5)
    test_opt.step()

    A_norm = test_m.vis_injector.adapters[0].A.weight.norm().item()
    B_norm = test_m.vis_injector.adapters[0].B.weight.norm().item()

    with torch.no_grad():
        preds = (sim_fus / CONFIG['temperature']).argmax(dim=-1)
        acc   = (preds == lbls).float().mean().item() * 100

    # Status
    if prev_loss is None:
        status = "start"
    elif L.item() < prev_loss - 0.01:
        status = "↓ improving"
    elif L.item() > prev_loss + 0.05:
        status = "↑ spike"
    else:
        status = "→ flat"

    print(f"{step+1:<6} {L.item():>8.4f} {acc:>7.1f}% "
          f"{A_norm:>10.6f} {B_norm:>10.6f}  {status}")

    prev_loss = L.item()

# Final diagnosis
print("\n--- Diagnosis ---")
b_final = test_m.vis_injector.adapters[0].B.weight.norm().item()
print(f"B_norm: 0.0 → {b_final:.6f}  "
      f"{'✓ adapter learned' if b_final > 0.001 else '✗ adapter dead'}")

del test_m, test_opt, _fresh_loader, _ds
torch.cuda.empty_cache()
gc.collect()

=== Quick Adapter Learning Test ===

Before training:
  A norm = 0.557487  (should be ~0.08)
  B norm = 0.000000  (should be 0.000000)

Client 9: 48 samples, 18 classes, 6 batches/epoch

Step       Loss      Acc     A_norm     B_norm Status
------------------------------------------------------------
1        0.8359    87.5%   0.557487   0.016625  start
2        1.1939    75.0%   0.558015   0.024919  ↑ spike
3        0.9816    87.5%   0.558598   0.032060  ↓ improving
4        0.8445   100.0%   0.558853   0.038415  ↓ improving
5        0.9789   100.0%   0.559201   0.044182  ↑ spike
6        1.1308    87.5%   0.559646   0.049217  ↑ spike

--- Diagnosis ---
B_norm: 0.0 → 0.049217  ✓ adapter learned


0

In [6]:
# ============================================================
# PART 5 (UPDATED): Run All Three Datasets
# OxfordPets loads from Drive (no retraining)
# EuroSAT and DTD train fresh
# ============================================================

import time

SAVE_DIR = os.path.join(DRIVE_CACHE_DIR, 'saved_models')
os.makedirs(SAVE_DIR, exist_ok=True)


# ── Save / load helpers ──────────────────────────────────────

def save_experiment(mode_ds, history,
                    client_models, dyn_gamma=None,
                    config=CONFIG):
    path = os.path.join(SAVE_DIR, f'blend_{mode_ds}.pt')
    states = [
        {'agg'  : m.get_aggregatable_params(),
         'local': m.get_local_params()}
        for m in client_models
    ]
    payload = {
        'mode'         : mode_ds,
        'config'       : config,
        'history'      : history,
        'client_states': states,
    }
    if dyn_gamma is not None:
        payload['dyn_gamma_state'] = \
            dyn_gamma.state_dict()
    torch.save(payload, path)
    mb = os.path.getsize(path) / 1e6
    print(f"Saved [{mode_ds}] → {path} ({mb:.1f} MB)")


def load_experiment(mode_ds):
    path = os.path.join(SAVE_DIR, f'blend_{mode_ds}.pt')
    if not os.path.exists(path):
        print(f"No saved model: {path}")
        return None, None, None
    print(f"Loading [{mode_ds}]...")
    p = torch.load(path, map_location=DEVICE)

    cfg    = p['config']
    hist   = p['history']
    states = p['client_states']

    models = [
        BLENDClientModel(clip_model, cfg).to(DEVICE)
        for _ in range(cfg['num_clients'])
    ]
    for model, state in zip(models, states):
        model.set_aggregatable_params(state['agg'])
        model.set_local_params(state['local'])

    dyn_g = None
    if 'dyn_gamma_state' in p:
        dyn_g = DynamicGammaModule(cfg).to(DEVICE)
        dyn_g.load_state_dict(p['dyn_gamma_state'])

    if hist['h_mean']:
        bi = int(np.argmax(hist['h_mean']))
        print(f"  Best  H: {hist['h_mean'][bi]:.2f}% "
              f"(round {hist['round'][bi]})")
        print(f"  Final H: {hist['h_mean'][-1]:.2f}%")
    return hist, models, dyn_g


def check_saved():
    print(f"\nSaved models in {SAVE_DIR}:")
    files = [f for f in os.listdir(SAVE_DIR)
             if f.endswith('.pt')]
    if not files:
        print("  None yet.")
        return
    for f in sorted(files):
        full = os.path.join(SAVE_DIR, f)
        mb   = os.path.getsize(full) / 1e6
        try:
            p    = torch.load(full, map_location='cpu')
            h    = p['history']['h_mean']
            best = max(h) if h else 0
            rds  = p['history']['round']
            last = rds[-1] if rds else 0
            print(f"  {f:<40} {mb:.1f}MB  "
                  f"rounds={last}  bestH={best:.2f}%")
        except Exception as e:
            print(f"  {f:<40} {mb:.1f}MB  (unreadable)")


check_saved()


# ============================================================
# Dataset preparation helper
# Packages everything needed for training
# into a single dict so we can loop cleanly
# ============================================================

def prepare_dataset_context(
    dataset_name, config, clip_preprocess
):
    """
    Load, split, preprocess and cache one dataset.
    Returns a context dict with all loaders,
    text embeddings, entropies etc. ready for
    run_federated_training.
    """
    # Cache dir for this dataset
    ds_cache = os.path.join(
        DRIVE_CACHE_DIR,
        f'{dataset_name}_cache'
    )
    os.makedirs(ds_cache, exist_ok=True)

    # ── Load raw data ─────────────────────────────────────
    (raw_train, raw_test,
     class_names, num_classes,
     train_labels, test_labels) = load_raw_dataset(
        dataset_name
    )

    # ── Base / novel split ────────────────────────────────
    split_file = os.path.join(
        DRIVE_CACHE_DIR,
        f'{dataset_name}_class_split.pt'
    )
    if os.path.exists(split_file):
        split         = torch.load(split_file)
        base_classes  = split['base']
        novel_classes = split['novel']
        print(f"  Split loaded: "
              f"{len(base_classes)} base / "
              f"{len(novel_classes)} novel")
    else:
        rng       = random.Random(SEED)
        all_ids   = list(range(num_classes))
        rng.shuffle(all_ids)
        nb        = num_classes // 2
        base_classes  = sorted(all_ids[:nb])
        novel_classes = sorted(all_ids[nb:])
        torch.save(
            {'base' : base_classes,
             'novel': novel_classes},
            split_file
        )
        print(f"  Split created: "
              f"{len(base_classes)} base / "
              f"{len(novel_classes)} novel")

    # ── Few-shot + Dirichlet ──────────────────────────────
    few_shot_idx = sample_few_shot(
        train_labels, base_classes,
        config['num_shots']
    )
    client_indices, client_class_counts = dirichlet_split(
        train_labels, few_shot_idx,
        base_classes,
        config['num_clients'],
        config['alpha_dir']
    )
    client_entropies = [
        compute_client_entropy(cc)
        for cc in client_class_counts
    ]
    client_remaps, client_local_classes = \
        build_label_remap(client_class_counts)

    print(f"  Few-shot pool: {len(few_shot_idx)}")
    print(f"  Entropy range: "
          f"{min(client_entropies):.3f} – "
          f"{max(client_entropies):.3f}")

    # ── Preprocess & cache ────────────────────────────────
    for cid, idxs in enumerate(client_indices):
        if len(idxs) == 0:
            continue
        preprocess_and_save_split(
            raw_train, idxs, train_labels,
            os.path.join(ds_cache, f'client_{cid}'),
            clip_preprocess,
            desc=f"{dataset_name}/client_{cid}"
        )

    base_test_idx  = [i for i, l
                      in enumerate(test_labels)
                      if l in base_classes]
    novel_test_idx = [i for i, l
                      in enumerate(test_labels)
                      if l in novel_classes]

    preprocess_and_save_split(
        raw_test, base_test_idx, test_labels,
        os.path.join(ds_cache, 'base_test'),
        clip_preprocess,
        desc=f"{dataset_name}/base_test"
    )
    preprocess_and_save_split(
        raw_test, novel_test_idx, test_labels,
        os.path.join(ds_cache, 'novel_test'),
        clip_preprocess,
        desc=f"{dataset_name}/novel_test"
    )

    # ── Build DataLoaders ─────────────────────────────────
    client_loaders = []
    for cid in range(config['num_clients']):
        cache_path = os.path.join(
            ds_cache, f'client_{cid}'
        )
        if not os.path.exists(cache_path) \
                or len(client_indices[cid]) == 0:
            client_loaders.append(None)
            continue
        ds     = DriveTensorDataset(
            cache_path,
            label_remap=client_remaps[cid]
        )
        loader = DataLoader(
            ds,
            batch_size  = config['batch_size'],
            shuffle     = True,
            num_workers = 0,
            pin_memory  = True,
        )
        client_loaders.append(loader)

    base_test_loader = DataLoader(
        DriveTensorDataset(
            os.path.join(ds_cache, 'base_test'),
            label_remap=None
        ),
        batch_size=64, shuffle=False, num_workers=0
    )
    novel_test_loader = DataLoader(
        DriveTensorDataset(
            os.path.join(ds_cache, 'novel_test'),
            label_remap=None
        ),
        batch_size=64, shuffle=False, num_workers=0
    )

    # ── Text embeddings ───────────────────────────────────
    base_text_emb  = get_text_embeddings(
        class_names, base_classes,  DEVICE
    )
    novel_text_emb = get_text_embeddings(
        class_names, novel_classes, DEVICE
    )

    client_text_embs = []
    for cid, counts in enumerate(client_class_counts):
        present = sorted(
            [c for c, n in counts.items() if n > 0]
        )
        if not present:
            client_text_embs.append(None)
            continue
        emb = get_text_embeddings(
            class_names, present, DEVICE
        )
        client_text_embs.append(emb)

    # Free raw data
    del raw_train, raw_test
    gc.collect()

    return {
        'dataset_name'       : dataset_name,
        'class_names'        : class_names,
        'num_classes'        : num_classes,
        'base_classes'       : base_classes,
        'novel_classes'      : novel_classes,
        'client_indices'     : client_indices,
        'client_class_counts': client_class_counts,
        'client_entropies'   : client_entropies,
        'client_remaps'      : client_remaps,
        'client_loaders'     : client_loaders,
        'base_test_loader'   : base_test_loader,
        'novel_test_loader'  : novel_test_loader,
        'base_text_emb'      : base_text_emb,
        'novel_text_emb'     : novel_text_emb,
        'client_text_embs'   : client_text_embs,
    }


# ============================================================
# TRAINING FLAGS
# Set False to load from Drive instead of retraining
# ============================================================

RETRAIN = {
    'OxfordPets_static'  : False,  # ← already done
    'OxfordPets_dynamic' : False,  # ← already done
    'EuroSAT_static'     : True,   # ← new
    'EuroSAT_dynamic'    : True,   # ← new
    'DTD_static'         : True,   # ← new
    'DTD_dynamic'        : True,   # ← new
}

DATASETS = ['OxfordPets', 'EuroSAT', 'DTD']

# Storage for all results
ALL_RESULTS = {}


# ============================================================
# Main loop over datasets
# ============================================================

for ds_name in DATASETS:

    print(f"\n{'#'*60}")
    print(f"#  DATASET: {ds_name}")
    print(f"{'#'*60}")

    # ── Skip preprocessing for OxfordPets ────────────────
    # (already cached from previous runs)
    if ds_name == 'OxfordPets':
        print("  OxfordPets: loading from Drive cache...")
        # We still need context for eval
        # but skip retraining
        ctx = None   # not needed since we load saved models
    else:
        ctx = prepare_dataset_context(
            ds_name, CONFIG, clip_preprocess
        )

    # ── Run static and dynamic ────────────────────────────
    ds_results = {}

    for mode in ['static', 'dynamic']:
        key = f'{ds_name}_{mode}'

        if not RETRAIN.get(key, True):
            # Load from Drive
            hist, models, dyn_g = load_experiment(key)
            if hist is not None:
                ds_results[mode] = {
                    'history'  : hist,
                    'dyn_gamma': dyn_g,
                }
                print(f"  [{key}] loaded ✓")
                continue
            else:
                print(f"  [{key}] not found, "
                      f"will train...")

        # ── For OxfordPets we need context too ───────────
        if ds_name == 'OxfordPets' and ctx is None:
            ctx = prepare_dataset_context(
                ds_name, CONFIG, clip_preprocess
            )

        # ── Inject context into global namespace ─────────
        # run_federated_training uses these globals
        # (same pattern as before)
        client_loaders      = ctx['client_loaders']
        client_indices      = ctx['client_indices']
        client_class_counts = ctx['client_class_counts']
        client_remaps       = ctx['client_remaps']
        client_local_classes= [
            sorted([c for c, n in cc.items() if n > 0])
            for cc in ctx['client_class_counts']
        ]
        client_entropies    = ctx['client_entropies']
        client_text_embs    = ctx['client_text_embs']
        base_test_loader    = ctx['base_test_loader']
        novel_test_loader   = ctx['novel_test_loader']
        base_text_emb       = ctx['base_text_emb']
        novel_text_emb      = ctx['novel_text_emb']
        BASE_CLASSES        = ctx['base_classes']
        NOVEL_CLASSES       = ctx['novel_classes']
        CLASS_NAMES         = ctx['class_names']

        # ── Train ─────────────────────────────────────────
        hist, models, dyn_g = run_federated_training(
            mode   = mode,
            config = CONFIG,
        )

        # ── Save ──────────────────────────────────────────
        save_experiment(key, hist, models, dyn_g)

        ds_results[mode] = {
            'history'  : hist,
            'dyn_gamma': dyn_g,
        }

        torch.cuda.empty_cache()
        gc.collect()

    ALL_RESULTS[ds_name] = ds_results
    print(f"\n  {ds_name} complete.")


# ============================================================
# Cross-dataset summary table
# ============================================================

print("\n\n" + "="*72)
print("CROSS-DATASET SUMMARY")
print("="*72)
print(f"\n{'Dataset':<14} {'Method':<18} "
      f"{'BestRnd':>7} {'BestH':>7} "
      f"{'FinalPer':>10} {'FinalGen':>10} "
      f"{'FinalH':>8} {'Drop':>7}")
print("-"*80)

for ds_name in DATASETS:
    if ds_name not in ALL_RESULTS:
        continue
    for mode, label in [
        ('static',  'Static BLEND'),
        ('dynamic', 'DG-BLEND-L'),
    ]:
        if mode not in ALL_RESULTS[ds_name]:
            continue
        hist = ALL_RESULTS[ds_name][mode]['history']
        if not hist['h_mean']:
            continue
        bi     = int(np.argmax(hist['h_mean']))
        best_h = hist['h_mean'][bi]
        best_r = hist['round'][bi]
        fin_p  = hist['per_acc'][-1]
        fin_g  = hist['gen_acc'][-1]
        fin_h  = hist['h_mean'][-1]
        drop   = best_h - fin_h

        print(f"  {ds_name:<12} {label:<18} "
              f"{best_r:>7} {best_h:>7.2f} "
              f"{fin_p:>10.2f} {fin_g:>10.2f} "
              f"{fin_h:>8.2f} {drop:>6.2f}pp")
    print()

print("="*72)
print("Run Part 6 for visualizations.")


Saved models in /content/drive/MyDrive/blend_oxfordpets_cache/saved_models:
  None yet.

############################################################
#  DATASET: OxfordPets
############################################################
  OxfordPets: loading from Drive cache...
No saved model: /content/drive/MyDrive/blend_oxfordpets_cache/saved_models/blend_OxfordPets_static.pt
  [OxfordPets_static] not found, will train...

Loading OxfordPets...
  Train: 3680 samples
  Test : 3669  samples
  Classes: 37
  Split loaded: 18 base / 19 novel
  Few-shot pool: 288
  Entropy range: 0.000 – 0.964
  [OxfordPets/client_0] Cached ✓
  [OxfordPets/client_1] Cached ✓
  [OxfordPets/client_2] Cached ✓
  [OxfordPets/client_3] Cached ✓
  [OxfordPets/client_4] Cached ✓
  [OxfordPets/client_5] Cached ✓
  [OxfordPets/client_6] Cached ✓
  [OxfordPets/client_7] Cached ✓
  [OxfordPets/client_8] Cached ✓
  [OxfordPets/client_9] Cached ✓
  [OxfordPets/base_test] Cached ✓
  [OxfordPets/novel_test] Cached ✓

  Mod

KeyboardInterrupt: 

In [ ]:
# ============================================================
# RUN THIS AFTER TRAINING COMPLETES
# Extracts all dynamic gamma values and analysis
# ============================================================

print("=" * 60)
print("DYNAMIC GAMMA ANALYSIS")
print("=" * 60)

# ── 1. Gamma trajectory per round (mean across clients) ──────
print("\n--- Mean γ per Round ---")
print(f"{'Round':>6} {'Mean_γ':>8} {'Std_γ':>8} "
      f"{'Min_γ':>8} {'Max_γ':>8} {'Per':>8} "
      f"{'Gen':>8} {'H':>8}")
print("-" * 65)

for i, rnd in enumerate(history_dynamic['round']):
    # Get gamma values for this round index
    # gamma_per_client[cid][round_idx]
    round_idx = rnd - 1
    gammas_this_round = []
    for cid in range(CONFIG['num_clients']):
        traj = history_dynamic['gamma_per_client'][cid]
        if len(traj) > round_idx:
            gammas_this_round.append(traj[round_idx])

    if gammas_this_round:
        mean_g = np.mean(gammas_this_round)
        std_g  = np.std(gammas_this_round)
        min_g  = np.min(gammas_this_round)
        max_g  = np.max(gammas_this_round)
    else:
        mean_g = std_g = min_g = max_g = 0.0

    per = history_dynamic['per_acc'][i]
    gen = history_dynamic['gen_acc'][i]
    h   = history_dynamic['h_mean'][i]

    print(f"{rnd:>6} {mean_g:>8.4f} {std_g:>8.4f} "
          f"{min_g:>8.4f} {max_g:>8.4f} {per:>8.2f} "
          f"{gen:>8.2f} {h:>8.2f}")


# ── 2. Per-client gamma at key rounds ────────────────────────
print("\n\n--- Per-Client γ at Key Rounds ---")
print(f"{'Client':>7} {'H̃':>6} {'Samples':>8} "
      f"{'γ(R1)':>8} {'γ(R8)':>8} "
      f"{'γ(R16)':>8} {'γ(R25)':>8} {'Trend':>10}")
print("-" * 70)

for cid in range(CONFIG['num_clients']):
    traj    = history_dynamic['gamma_per_client'][cid]
    n       = len(client_indices[cid])
    H       = client_entropies[cid]

    if len(traj) == 0:
        continue

    g1  = traj[0]
    g8  = traj[min(7,  len(traj)-1)]
    g16 = traj[min(15, len(traj)-1)]
    g25 = traj[-1]

    if   g25 > g1 + 0.005: trend = "↑ rising"
    elif g25 < g1 - 0.005: trend = "↓ falling"
    else:                   trend = "→ stable"

    print(f"  {cid:>5} {H:>6.3f} {n:>8} "
          f"{g1:>8.4f} {g8:>8.4f} "
          f"{g16:>8.4f} {g25:>8.4f} {trend:>10}")

# Summary stats
print()
for r_idx, r_name in [(0,'R=1'), (7,'R=8'), (14,'R=15'), (-1,'R=25')]:
    vals = [
        history_dynamic['gamma_per_client'][c][r_idx]
        for c in range(CONFIG['num_clients'])
        if len(history_dynamic['gamma_per_client'][c]) > abs(r_idx)
    ]
    if vals:
        print(f"  {r_name}: mean={np.mean(vals):.4f}  "
              f"std={np.std(vals):.4f}  "
              f"min={np.min(vals):.4f}  "
              f"max={np.max(vals):.4f}")


# ── 3. Tau parameter evolution ───────────────────────────────
print("\n\n--- Tau Parameter Evolution ---")
if history_dynamic['tau_e_history']:
    print(f"{'Round':>6} {'τ_e':>8} {'τ_d':>8} {'bias':>8} "
          f"{'γ_formula':>12}")
    print("-" * 50)

    # Show at key rounds
    tau_rounds = [0, 4, 9, 14, 19, 24]
    for i in tau_rounds:
        if i >= len(history_dynamic['tau_e_history']):
            continue
        te  = history_dynamic['tau_e_history'][i]
        td  = history_dynamic['tau_d_history'][i]
        b   = history_dynamic['bias_history'][i]
        rnd = i + 1
        # Example gamma for a skewed client H̃=0.1
        import torch
        example_logit = te * (1-0.1) + td * 0.0 - b
        example_gamma = torch.sigmoid(
            torch.tensor(example_logit)
        ).item()
        print(f"  {rnd:>4} {te:>8.4f} {td:>8.4f} "
              f"{b:>8.4f} {example_gamma:>12.4f}")

    print(f"\n  τ_e change: "
          f"{history_dynamic['tau_e_history'][0]:.4f} → "
          f"{history_dynamic['tau_e_history'][-1]:.4f} "
          f"(Δ={history_dynamic['tau_e_history'][-1]-history_dynamic['tau_e_history'][0]:+.4f})")
    print(f"  τ_d change: "
          f"{history_dynamic['tau_d_history'][0]:.4f} → "
          f"{history_dynamic['tau_d_history'][-1]:.4f} "
          f"(Δ={history_dynamic['tau_d_history'][-1]-history_dynamic['tau_d_history'][0]:+.4f})")
    print(f"  bias change: "
          f"{history_dynamic['bias_history'][0]:.4f} → "
          f"{history_dynamic['bias_history'][-1]:.4f} "
          f"(Δ={history_dynamic['bias_history'][-1]-history_dynamic['bias_history'][0]:+.4f})")
else:
    print("  No tau history recorded")


# ── 4. Entropy vs gamma correlation ──────────────────────────
print("\n\n--- Entropy vs Final γ Correlation ---")
final_gammas = [
    history_dynamic['gamma_per_client'][c][-1]
    for c in range(CONFIG['num_clients'])
    if history_dynamic['gamma_per_client'][c]
]
ents = [
    client_entropies[c]
    for c in range(CONFIG['num_clients'])
    if history_dynamic['gamma_per_client'][c]
]
if len(final_gammas) > 1:
    corr = np.corrcoef(ents, final_gammas)[0, 1]
    print(f"  Pearson ρ(H̃_n, γ_final) = {corr:.4f}")
    print(f"  Interpretation: "
          f"{'negative ✓ — high entropy → low γ (correct)' if corr < 0 else 'positive — check formula'}")

    # Show sorted
    paired = sorted(zip(ents, final_gammas))
    print(f"\n  Sorted by entropy (low→high):")
    print(f"  {'H̃':>6}  {'γ_final':>8}  {'Expected':>10}")
    for e, g in paired:
        expected = "high γ" if e < 0.3 else \
                   "low γ"  if e > 0.7 else "mid γ"
        marker = "✓" if (e < 0.3 and g > 0.08) or \
                        (e > 0.7 and g < 0.08) else "?"
        print(f"  {e:>6.3f}  {g:>8.4f}  "
              f"{expected:>10}  {marker}")


# ── 5. Key numbers for paper ──────────────────────────────────
print("\n\n--- KEY NUMBERS FOR PAPER ---")
print("\nDG-BLEND-P:")
if history_dynamic['h_mean']:
    best_i = int(np.argmax(history_dynamic['h_mean']))
    print(f"  Best round  : {history_dynamic['round'][best_i]}")
    print(f"  Best Per    : {history_dynamic['per_acc'][best_i]:.2f}%")
    print(f"  Best Gen    : {history_dynamic['gen_acc'][best_i]:.2f}%")
    print(f"  Best H-Mean : {history_dynamic['h_mean'][best_i]:.2f}%")
    print(f"  Final Per   : {history_dynamic['per_acc'][-1]:.2f}%")
    print(f"  Final Gen   : {history_dynamic['gen_acc'][-1]:.2f}%")
    print(f"  Final H-Mean: {history_dynamic['h_mean'][-1]:.2f}%")
    print(f"  Max H drop  : "
          f"{max(history_dynamic['h_mean'])-min(history_dynamic['h_mean']):.2f}%")

print("\nStatic BLEND:")
if history_static['h_mean']:
    best_i = int(np.argmax(history_static['h_mean']))
    print(f"  Best round  : {history_static['round'][best_i]}")
    print(f"  Best Per    : {history_static['per_acc'][best_i]:.2f}%")
    print(f"  Best Gen    : {history_static['gen_acc'][best_i]:.2f}%")
    print(f"  Best H-Mean : {history_static['h_mean'][best_i]:.2f}%")
    print(f"  Final Per   : {history_static['per_acc'][-1]:.2f}%")
    print(f"  Final Gen   : {history_static['gen_acc'][-1]:.2f}%")
    print(f"  Final H-Mean: {history_static['h_mean'][-1]:.2f}%")
    print(f"  Max H drop  : "
          f"{max(history_static['h_mean'])-min(history_static['h_mean']):.2f}%")

print("\nDeltas (Dynamic - Static):")
if history_dynamic['h_mean'] and history_static['h_mean']:
    print(f"  Δ Best H    : "
          f"{max(history_dynamic['h_mean'])-max(history_static['h_mean']):+.2f}%")
    print(f"  Δ Final H   : "
          f"{history_dynamic['h_mean'][-1]-history_static['h_mean'][-1]:+.2f}%")
    print(f"  Δ Stability : "
          f"{(max(history_static['h_mean'])-min(history_static['h_mean']))-(max(history_dynamic['h_mean'])-min(history_dynamic['h_mean'])):+.2f}% "
          f"(positive = dynamic more stable)")